In [4]:
# ==========================================
#             DATA LOADING
# ==========================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn import set_config
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. BẬT TÍNH NĂNG XUẤT BẢNG CỦA SCIKIT-LEARN
set_config(transform_output="pandas")

# 2. Đọc dữ liệu với đường dẫn chính xác của bạn
df = pd.read_csv('../../../data/insurance.csv')

# 3. Phân tách biến độc lập X và biến mục tiêu y
X = df.drop('charges', axis=1)
y = df['charges']

# 4. Chia tập Train/Test theo tỷ lệ 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Thiết lập các bộ tiền xử lý
numeric_features = ['age', 'bmi', 'children']
categorical_features = ['sex', 'region']
binary_features = ['smoker']

# Xây dựng ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), categorical_features),
        ('bin', OrdinalEncoder(categories=[['no', 'yes']]), binary_features)
    ])

# 6. CHẠY THỬ VÀ HIỂN THỊ BẢNG
X_train_processed = preprocessor.fit_transform(X_train)

display(X_train_processed.head())

,num__age,num__bmi,num__children,cat__sex_male,cat__region_northwest,cat__region_southeast,cat__region_southwest,bin__smoker
560,0.472227,-1.756525,0.734336,0.0,1.0,0.0,0.0,0.0
1285,0.543313,-1.033082,-0.911192,0.0,0.0,0.0,0.0,0.0
1142,0.898745,-0.943687,-0.911192,0.0,0.0,1.0,0.0,0.0
969,-0.025379,0.622393,3.202629,0.0,0.0,1.0,0.0,0.0
486,1.040918,-1.504893,1.557100,0.0,1.0,0.0,0.0,0.0


In [5]:
# ==========================================
#                MODELING
# ==========================================

# 1. Đóng gói Preprocessor và Model Ridge vào Full Pipeline
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge())
])

# 2. Thiết lập Hyperparameter Tuning
param_grid = {
    'regressor__alpha': [0.01, 0.1, 1.0, 10.0, 50.0, 100.0]
}

# Sử dụng GridSearchCV thực hiện 5-Fold Cross Validation
grid_search = GridSearchCV(full_pipeline, param_grid, cv=5, scoring='r2', n_jobs=-1)

print("Đang huấn luyện và tìm tham số tối ưu (5-Fold CV)...")
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print(f"Tham số Alpha tối ưu tìm được: {grid_search.best_params_['regressor__alpha']}\n")

# 3. Dự đoán và Đánh giá trên tập Test
y_pred = best_model.predict(X_test)

print("="*45)
print("KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH (RIDGE REGRESSION)")
print("="*45)
print(f"R2 Score (Trên tập Train - 5-Fold CV):  {grid_search.best_score_:.4f}")
print(f"R2 Score (Trên tập Test thực tế):      {r2_score(y_test, y_pred):.4f}")
print(f"MAE (Sai số tuyệt đối):                ${mean_absolute_error(y_test, y_pred):.2f}")
print(f"RMSE (Sai số căn bậc hai):             ${np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
print("="*45)

Đang huấn luyện và tìm tham số tối ưu (5-Fold CV)...
Tham số Alpha tối ưu tìm được: 1.0

KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH (RIDGE REGRESSION)
R2 Score (Trên tập Train - 5-Fold CV):  0.7332
R2 Score (Trên tập Test thực tế):      0.7833
MAE (Sai số tuyệt đối):                $4193.20
RMSE (Sai số căn bậc hai):             $5800.46
